In [11]:
# ----- Setup -----
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re  #-------

# ----- Load Zara 2 Data -----
df = pd.read_csv("converted_zara_2.csv")
x_mean, x_std = df['x'].mean(), df['x'].std()
y_mean, y_std = df['y'].mean(), df['y'].std()
df['x'] = (df['x'] - x_mean) / x_std
df['y'] = (df['y'] - y_mean) / y_std

SEQ_LEN = 8
PRED_LEN = 12

# ----- Dataset Definition -----
class TrajectoryDataset(Dataset):
    def __init__(self, trajectories):
        self.trajectories = trajectories

    def __len__(self):
        return len(self.trajectories)

    def __getitem__(self, idx):
        obs, fut = self.trajectories[idx]
        return torch.tensor(obs, dtype=torch.float32), torch.tensor(fut, dtype=torch.float32)

# ----- Rebuild Zara 2 Trajectories -----
trajectories = []
for pid, person_df in df.groupby("person_id"):
    person_df = person_df.sort_values("frame_id")
    coords = person_df[['x', 'y']].values
    for i in range(len(coords) - SEQ_LEN - PRED_LEN):
        obs = coords[i:i+SEQ_LEN]
        fut = coords[i+SEQ_LEN:i+SEQ_LEN+PRED_LEN]
        trajectories.append((obs, fut))

data_loader = DataLoader(TrajectoryDataset(trajectories), batch_size=64, shuffle=False)

# ----- LSTM Model Definition (same as before) -----
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_len, num_layers, dropout, bidirectional):
        super(LSTMModel, self).__init__()
        self.bidirectional = bidirectional
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True, bidirectional=bidirectional
        )
        direction_multiplier = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_size * direction_multiplier, 2 * output_len)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        hn = torch.cat((hn[-2], hn[-1]), dim=1) if self.bidirectional else hn[-1]
        out = self.fc(hn)
        return out.view(-1, self.fc.out_features // 2, 2)

# ----- Evaluate All Matching Models in Directory -----
results = []
criterion = nn.MSELoss()

#model_files = [f for f in os.listdir('.') if re.match(r"lstm_zara_exp_\\d+\\.pth", f)]  #-------
model_files = [f for f in os.listdir('.') if re.match(r"lstm_zara_exp_\d+\.pth", f)]
for model_file in sorted(model_files, key=lambda x: int(re.findall(r"\d+", x)[0])):  #-------
    exp_id = int(re.findall(r"\d+", model_file)[0])  #-------

    # Load corresponding metadata file (assumes JSON was lost, so fallback)
    metadata_path = f"metadata_exp_{exp_id}.json"  #-------
    if not os.path.exists(metadata_path):
        print(f"Skipping {model_file}: missing metadata")  #-------
        continue

    with open(metadata_path, 'r') as f:
        row = json.load(f)  #-------

    model = LSTMModel(
        input_size=2,
        hidden_size=row["hidden_size"],
        output_len=row["pred_len"],
        num_layers=row["num_layers"],
        dropout=row["dropout"],
        bidirectional=row["bidirectional"]
    )
    model.load_state_dict(torch.load(model_file))
    model.eval()

    total_loss = 0
    with torch.no_grad():
        for obs, fut in data_loader:
            pred = model(obs)
            loss = criterion(pred, fut)
            total_loss += loss.item()

    avg_loss = total_loss / len(data_loader)
    row["zara2_loss"] = avg_loss
    row["experiment_id"] = exp_id
    results.append(row)  #-------

# ----- Save Results Back to JSON (optional) -----
pd.DataFrame(results).to_json("results_metadata_rebuilt.json", orient="records", indent=2)  #-------


Skipping lstm_zara_exp_0.pth: missing metadata
Skipping lstm_zara_exp_1.pth: missing metadata
Skipping lstm_zara_exp_2.pth: missing metadata
Skipping lstm_zara_exp_3.pth: missing metadata
Skipping lstm_zara_exp_4.pth: missing metadata
Skipping lstm_zara_exp_5.pth: missing metadata
Skipping lstm_zara_exp_6.pth: missing metadata
Skipping lstm_zara_exp_7.pth: missing metadata
Skipping lstm_zara_exp_8.pth: missing metadata
Skipping lstm_zara_exp_9.pth: missing metadata
Skipping lstm_zara_exp_10.pth: missing metadata
Skipping lstm_zara_exp_11.pth: missing metadata
Skipping lstm_zara_exp_12.pth: missing metadata
Skipping lstm_zara_exp_13.pth: missing metadata
Skipping lstm_zara_exp_14.pth: missing metadata
Skipping lstm_zara_exp_15.pth: missing metadata
Skipping lstm_zara_exp_16.pth: missing metadata
Skipping lstm_zara_exp_17.pth: missing metadata
Skipping lstm_zara_exp_18.pth: missing metadata
Skipping lstm_zara_exp_19.pth: missing metadata
Skipping lstm_zara_exp_20.pth: missing metadata
Sk

In [12]:
print(f"Found {len(model_files)} model files.")
print(model_files)

Found 43 model files.
['lstm_zara_exp_33.pth', 'lstm_zara_exp_38.pth', 'lstm_zara_exp_35.pth', 'lstm_zara_exp_10.pth', 'lstm_zara_exp_5.pth', 'lstm_zara_exp_39.pth', 'lstm_zara_exp_30.pth', 'lstm_zara_exp_32.pth', 'lstm_zara_exp_20.pth', 'lstm_zara_exp_25.pth', 'lstm_zara_exp_21.pth', 'lstm_zara_exp_29.pth', 'lstm_zara_exp_28.pth', 'lstm_zara_exp_17.pth', 'lstm_zara_exp_40.pth', 'lstm_zara_exp_6.pth', 'lstm_zara_exp_34.pth', 'lstm_zara_exp_41.pth', 'lstm_zara_exp_18.pth', 'lstm_zara_exp_1.pth', 'lstm_zara_exp_9.pth', 'lstm_zara_exp_22.pth', 'lstm_zara_exp_11.pth', 'lstm_zara_exp_27.pth', 'lstm_zara_exp_36.pth', 'lstm_zara_exp_2.pth', 'lstm_zara_exp_13.pth', 'lstm_zara_exp_0.pth', 'lstm_zara_exp_3.pth', 'lstm_zara_exp_23.pth', 'lstm_zara_exp_7.pth', 'lstm_zara_exp_31.pth', 'lstm_zara_exp_26.pth', 'lstm_zara_exp_4.pth', 'lstm_zara_exp_16.pth', 'lstm_zara_exp_24.pth', 'lstm_zara_exp_19.pth', 'lstm_zara_exp_37.pth', 'lstm_zara_exp_15.pth', 'lstm_zara_exp_8.pth', 'lstm_zara_exp_42.pth', 'ls

In [13]:
# ----- Visualize Combined Results -----
results_df = pd.DataFrame(results)

# Ensure sorting works even if zara2_loss is missing in any row
results_df = results_df[results_df["zara2_loss"].notnull()].sort_values("zara2_loss")

plt.figure(figsize=(12, 6))
sns.barplot(data=results_df, x="experiment_id", y="zara2_loss", palette="viridis")
plt.title("Performance of LSTM Models on Zara 2 Dataset")
plt.xlabel("Experiment ID")
plt.ylabel("MSE Loss on Zara 2")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

KeyError: 'zara2_loss'